<h1 align = "center">
Rwanda Maize-Urea Trial
</h1>

<h3 align = "center">
Data Audit
</h3>

# Objectives

The data audit phase is intended to establish the structure, grain, quality and relationship of the datasets created during the Rwanda Maize-Urea field trial.

At the end of this audit exercise, we should be able to answer the following questions:

1. **What exactly is contained in each dataset?**

2. **What does a row represent in each dataset?**

3. **How are the datasets related?** and

4. **Are there any structural/data-quality issues that could affect subsequent analysis?**

We want to divide the data audit exercise into six major audits:

1. Data inventory
2. Structure and grain understanding
3. Data types identification
4. Keys, uniqueness and relationships
5. Missingness and completeness
6. Experimental-design and logical checks

Through these audit steps, we are interested in assessing:

- data dimensions
- variable structure
- data types
- missing values
- duplicates
- uniqueness of identifiers
- relationship between datasets
- experimental structure
- logical consistency of key variables


> This exercise does not involve cleaning the data just yet; We're just interested in **discovering and documenting problems**, not fixing them yet

At this point we've established that our project repository structure will be something like this:

    rwanda-maize-urea-analysis/
    │
    ├── 01_Data/
    │   ├── raw/
    │   ├── external/
    │   └── processed/
    │
    ├── 02_Notebooks/
    │
    ├── 03_Scripts/
    │
    ├── 04_Outputs/
    │   ├── figures/
    │   ├── tables/
    │   └── models/
    │
    ├── 05_Report/
    │
    ├── 06_Dashboard/
    │   ├── images/
    │   └── screenshots
    │
    └── README.md


Therefore, we should first have our original datasets 
- `Field_Level`
- `Plot_Level` and
- `Soil_PL`

in our data/raw folder.

 > We don't want to modify these raw files; That's important for reproducibility
 > Raw data must remain untouched - transformations should happen through code.

# Import Python Libraries

For now, we'll keep this deliberately small

In [11]:
import pandas as pd
import numpy as np

Later, we can introduce other libraries, including, but not limited to:
- `matplotlib`
- `seaborn`
- `scipy`
- `statsmodels`
- `sklearn`
- `geopandas`, etc.

But we're refraining from importing everything at the begining to maintain a professional outlook

Remember, a good analytical notebook introduces dependencies when they're actually needed.

## Load Datasets

We can opt to upload the datsets by typing in the path to the datasets each time or use a cleaner option:
    
- Since we have all the necessary datasets in one folder, we can opt for loading the data using a cleaner path

 - We'll need to define the dataset path first and then load each data frame using a shorter code each time we want to read. 

Alternatively, an even cleaner option would be to load the datasets using a  helper function in pathlib. 

We'll opt for this second approach instead.

In [12]:
from pathlib import Path

# We define the path to folder
data_path = Path(r"../01_Data/raw")

def load_csv (filename):
    return pd.read_csv(data_path/filename)

# Now we use it to access the datasets

field_df = load_csv("Field_Level.csv")
plot_df = load_csv("Plot_Level.csv")
soil_df = load_csv("Soil_PL.csv")

Before we start auditing the dataframes we have column by column, we want to establish the size of each dataset:

- We want to look at a few things that will point us to the direction of what to look for:

Essentially, we want to look at the shape of the respective datasets.

- The shape of a dataset helps us know the number of fields and records we have in the dataframe. 
- This way we can assess whether the dataframe is balanced (not having too many fields with too few records) or something of the sort.
- We'll create a small inventory table.
 

In [13]:
datasets = {
    "Field" : field_df,
    "Plot" : plot_df,
    "Soil" : soil_df   
}
   
inventory = []

for name, df in datasets.items():
    inventory.append({
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1]
    })

inventory_df = pd.DataFrame(inventory)

inventory_df

,dataset,rows,columns
0,Field,399,19
1,Plot,1598,27
2,Soil,736,5


## Column Audit


What we are interested to see here is whethere there exists any suspicious variables. 

We'll be looking for variables that have been stored wrongly. For example:

- numeric variables stored as string
- datetime variables stored as string
- string variables stored as object etc.

#### Field Dataset

Let's start by identifying the columns listed in each dataset to see if it matches what's in the codebook

In [14]:
field_df.columns

Index(['block_id', 'season', 'district', 'cell', 'agzone', 'meg_env', 'lat',
       'long', 'alt', 'landscape_pos', 'slope_pos', 'fert_rank', 'soil_color',
       'rockness', 'soil_depth', 'slope', 'trial_follower_gender',
       'trialfield_home_min', 'cult_land'],
      dtype='str')

Now we make the audit more usefull. We identify:
-  column names
- non-null counts
- data types

In [15]:
display(field_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 399 entries, 0 to 398
Data columns (total 19 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   block_id               399 non-null    str    
 1   season                 399 non-null    str    
 2   district               399 non-null    str    
 3   cell                   399 non-null    str    
 4   agzone                 399 non-null    str    
 5   meg_env                399 non-null    str    
 6   lat                    399 non-null    float64
 7   long                   399 non-null    float64
 8   alt                    399 non-null    float64
 9   landscape_pos          399 non-null    str    
 10  slope_pos              242 non-null    str    
 11  fert_rank              399 non-null    str    
 12  soil_color             399 non-null    str    
 13  rockness               242 non-null    str    
 14  soil_depth             242 non-null    str    
 15  slope            

None

##### Key results

Column                | Non-Null      | Expected Dtype  | Actual Dtype | Issue 
----------------------|---------------|-----------------|--------------|------
block_id              | 399           |string           |string        |
season                | 399           |string           |string        |
district              | 399           |string           |string        |
cell                  | 399           |string           |string        |
agzone                | 399           |string           |string        |
meg_env               | 399           |string           |string        |
lat                   | 399           |float            |float         |
long                  | 399           |float            |float         |
alt                   | 399           |float            |float         |
landscape_pos         | 399           |string           |string        |
slope_pos             | 242           |string           |string        |Substantial null records existing
fert_rank             | 399           |string           |string        |
soil_color            | 399           |string           |string        |
rockness              | 242           |string           |string        |Substantial null records existing
soil_depth            | 242           |string           |string        |Substantial null records existing
slope                 | 399           |integer          |integer       |
trial_follower_gender | 390           |string           |string        |A few null records existing
trialfield_home_min   | 118           |float            |float         |Substantial null records existing
cult_land             | 390           |float            |float         |A few null records existing


We can see that all columns in the field dataframe match columns from the codebook 

- Also, each column has been stored correctly, and

- They have a high proportion of non-null values, 

    - This suggests low percentage of missingness.

In [16]:
display(field_df.head())

,block_id,season,district,cell,agzone,meg_env,lat,long,alt,landscape_pos,slope_pos,fert_rank,soil_color,rockness,soil_depth,slope,trial_follower_gender,trialfield_home_min,cult_land
0,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,Rwamagana,Akanzu,Eastern Ridges,Low-Mid-Dry,-2.021072,30.334707,1638.04,Hills,Backslope,medium,red_less,no_rocks,shallow,3,Male,4.0,20.0
1,d8807f25-e675-49ae-87a3-d228eca23c72,21A,Rwamagana,Akanzu,Eastern Ridges,Low-Mid-Dry,-2.021166,30.334817,1622.55,Hills,Backslope,medium,red_more,less_rocky,shallow,4,Male,5.0,19.0
2,262efa71-6ff4-4e18-b6f1-0c3d293504c0,21A,Rwamagana,Akanzu,Eastern Ridges,Low-Mid-Dry,-2.020386,30.338258,1610.46,Hills,Backslope,medium,red_less,no_rocks,deep,3,Male,15.0,18.0
3,95c016dd-908a-4f07-9e49-926e9a9f4f19,21A,Rwamagana,Akanzu,Eastern Ridges,Low-Mid-Dry,-2.020589,30.337583,1620.42,Hills,Backslope,medium,red_less,no_rocks,deep,3,Male,3.0,18.0
4,1d5c61a0-ca6b-4db8-87c3-5ca46c7cad6d,21A,Rwamagana,Akanzu,Eastern Ridges,Low-Mid-Dry,-2.018119,30.338658,1614.13,Hills,Backslope,medium,black_less,less_rocky,shallow,4,Female,5.0,10.0


A glimpse of the dataframe indicates that the values are correctly stored for each field

#### Plot Dataset

Now let's do the same for the plot dataset

In [17]:
plot_df.columns

Index(['block_id', 'season', 'plot_id', 'treat_code', 'treat_details',
       'urea_rate_timing', 'crop', 'prior_season_crop', 'planting_date',
       'compost_tha', 'compost_quality', 'seed_variety', 'seeds_kgha',
       'dap_kgha', 'urea_planting_kgha', 'germ_count_date', 'germination_rate',
       'urea_V6_date', 'urea_V6_kgha', 'urea_V10_date', 'urea_V10_kgha',
       'urea_total_kgha', 'N_kgha', 'silking_date', 'R3_date', 'harvest_date',
       'yield_tha'],
      dtype='str')

In [18]:
  
display(plot_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 1598 entries, 0 to 1597
Data columns (total 27 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   block_id            1598 non-null   str    
 1   season              1598 non-null   str    
 2   plot_id             1598 non-null   str    
 3   treat_code          1598 non-null   str    
 4   treat_details       1598 non-null   str    
 5   urea_rate_timing    1598 non-null   str    
 6   crop                1598 non-null   str    
 7   prior_season_crop   1138 non-null   str    
 8   planting_date       1598 non-null   str    
 9   compost_tha         1598 non-null   int64  
 10  compost_quality     1590 non-null   str    
 11  seed_variety        1598 non-null   str    
 12  seeds_kgha          1598 non-null   int64  
 13  dap_kgha            1598 non-null   int64  
 14  urea_planting_kgha  1598 non-null   int64  
 15  germ_count_date     1598 non-null   str    
 16  germination_rate 

None

In [19]:
display(plot_df.head())

,block_id,season,plot_id,treat_code,treat_details,urea_rate_timing,crop,prior_season_crop,planting_date,compost_tha,...,urea_V6_date,urea_V6_kgha,urea_V10_date,urea_V10_kgha,urea_total_kgha,N_kgha,silking_date,R3_date,harvest_date,yield_tha
0,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-A,A,DAP 1Kg/Are + Urea 0.5Kg/Are at V6,Urea 0.5Kg/Are(V6),maize,maize,9/26/2020,8,...,11/11/2020,50,NaN,0,50,41,12/24/2020,1/13/2021,2/24/2021,4.52
1,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-B,B,DAP 1Kg/Are + Urea 0.5Kg/Are at Planting and U...,Urea1kg/Are(Planting+V6),maize,maize,9/26/2020,8,...,11/11/2020,50,NaN,0,100,64,12/24/2020,1/13/2021,2/24/2021,4.20
2,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-C,C,DAP 1Kg/Are + Urea 0.5Kg/Are at V6 + Urea 0.5K...,Urea 1kg/Are(V6 + V10),maize,maize,9/26/2020,8,...,11/11/2020,50,12/3/2020,50,100,64,12/24/2020,1/13/2021,2/24/2021,3.49
3,adfa31ad-95bf-453d-94f1-d163a97756ee,21A,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-D,D,DAP 1Kg/Are at planting+ Urea 1Kg/Are at V6,Urea 1kg/Are(V6),maize,maize,9/26/2020,8,...,11/11/2020,100,NaN,0,100,64,12/24/2020,1/13/2021,2/24/2021,3.58
4,d8807f25-e675-49ae-87a3-d228eca23c72,21A,d8807f25-e675-49ae-87a3-d228eca23c72-21A-A,A,DAP 1Kg/Are + Urea 0.5Kg/Are at V6,Urea 0.5Kg/Are(V6),maize,maize,9/26/2020,10,...,11/11/2020,50,NaN,0,50,41,12/24/2020,1/13/2021,2/24/2021,4.59


##### Key Results

Column             | Non-Null | Expected Dtype | Actual Dtype | Issue
-------------------|----------|----------------|--------------|----------  
block_id           | 1598     | string         | string       | 
season             | 1598     | string         | string       |
plot_id            | 1598     | string         | string       |
treat_code         | 1598     | string         | string       |
treat_details      | 1598     | string         | string       |
urea_rate_timing   | 1598     | string         | string       |
crop               | 1598     | string         | string       |
prior_season_crop  | 1138     | string         | string       |
planting_date      | 1598     | date           | string       | Incorrect data type stored
compost_tha        | 1598     | integer        | integer      |
compost_quality    | 1590     | string         | string       |
seed_variety       | 1598     | string         | string       |
seeds_kgha         | 1598     | integer        | integer      |
dap_kgha           | 1598     | integer        | integer      |
urea_planting_kgha | 1598     | integer        | integer      |
germ_count_date    | 1598     | date           | string       | Incorrect data type stored
germination_rate   | 1598     | integer        | integer      |
urea_V6_date       | 1598     | string         | string       |
urea_V6_kgha       | 1598     | integer        | integer      |
urea_V10_date      | 286      | date           | string       | Incorrect data type stored; substantial missing values
urea_V10_kgha      | 1598     | integer        | integer      |
urea_total_kgha    | 1598     | integer        | integer      |
N_kgha             | 1598     | integer        | integer      |
silking_date       | 1589     | date           | string       | Incorrect data type stored
R3_date            | 1592     | date           | string       | Incorrect data type stored; a few missing values
harvest_date       | 1598     | date           | string       | Incorrect data type stored
yield_tha          | 1598     | float          | float        |


- We realize that in the plot dataset there are a few fields with data stored in the wrong formats: 

    - *planting_date, germ_count_date, urea_V6_date, urea_V10_date, silking_date, R3_date and harvest_date* have been stored as string variables instead of datetime. 

- These highlight the priority areas for data cleaning. 

    - For instance, we now know we'll need to convert these variables to datetime for us to properly manipulate and analyze the data correctly.

#### Soil Dataset

- Now we do the same for the soil dataset

In [20]:
soil_df.columns

Index(['plot_id', 'pH', 'tot_N', 'org_C', 'sand'], dtype='str')

In [21]:
display(soil_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 736 entries, 0 to 735
Data columns (total 5 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   plot_id  736 non-null    str    
 1   pH       734 non-null    float64
 2   tot_N    0 non-null      float64
 3   org_C    0 non-null      float64
 4   sand     0 non-null      float64
dtypes: float64(4), str(1)
memory usage: 28.9 KB


None

In [22]:
display(soil_df.head())

,plot_id,pH,tot_N,org_C,sand
0,adfa31ad-95bf-453d-94f1-d163a97756ee-21A-A,5.21,NaN,NaN,NaN
1,d8807f25-e675-49ae-87a3-d228eca23c72-21A-A,5.75,NaN,NaN,NaN
2,262efa71-6ff4-4e18-b6f1-0c3d293504c0-21A-A,5.81,NaN,NaN,NaN
3,95c016dd-908a-4f07-9e49-926e9a9f4f19-21A-A,5.75,NaN,NaN,NaN
4,1d5c61a0-ca6b-4db8-87c3-5ca46c7cad6d-21A-A,6.13,NaN,NaN,NaN


##### Key Results

 Column  | Non-Null       | Expected Dtype | Actual Dtype  | Issue
---------|----------------|----------------|---------------|----------  
plot_id  | 736 non-null   | string         | string        |
pH       | 734 non-null   | float          | float         | A few missing values
tot_N    | 0 non-null     | float          | float         | Totally empty field
org_C    | 0 non-null     | float          | float         | Totally empty field
sand     | 0 non-null     | float          | float         | Totally empty field


- We can learn from the output above that *tot_N, org_C and sand* columns are completely empty

    - This calls for a need to augment soil data with recors from available databases to allow for analysis.

## Dataset Grain


### Field Grain

We want to establish the grain of each dataset
- We expect block_id in field dataset to identify a field per season
- Based on our understanding of the project, each row is supposed to represent one trial field in one season.

In [23]:
field_df["block_id"].nunique()

399

In [24]:
field_df.shape[0]

399

Comparing the two, we  realize the output is identical
- This is encouraging, and confirms our hypothesis that one rowrepresents one trial field in one season.
- We now check for whether there are duplicate rows in the field dataset

In [25]:
field_df["block_id"].duplicated().sum()

np.int64(0)

The output confirms there are no dupplicate records.
- But we have to ask whether block_id can repeat across seasons
    - We can't assume that block_id is necessarily the complete key
        - so we test block_id + season to uniquely identify field-season observation

In [27]:
field_df.duplicated(subset=["block_id", "season"]).sum()

np.int64(0)

No we can be certain there are no duplpicate block_id even by season.

### Plot Grain

We repeat the steps for plot dataset, now testing uniqueness by `plot_id`

In [29]:
plot_df["plot_id"].nunique()

1598

In [30]:
plot_df.shape[0]

1598

In [31]:
plot_df.duplicated(subset=["block_id","season","plot_id"]).sum()

np.int64(0)

Again, proof there are no duplicate records, each row represents one unique plot.

### The four-plots-per-fieldhypothesis

From the experiment, each field was supposedly divided into four plots
- It's important that we check whether the actual data supports this
    - This an important experimental-design check

In [32]:
plots_per_field = (
    plot_df.groupby(["block_id", "season"])["plot_id"]
        .nunique()
)

plots_per_field.value_counts().sort_index()

plot_id
2      1
3      4
4    396
Name: count, dtype: int64

The output suggests:
- 396 field-season combinations contain four plots
- 4 field-season combinations contain 3 plots, and
- 1 field-season combination contain 2 plots

### Treatment allocation check

The experiment supposedly contains four treatments
-  we want to see what is actually present

In [33]:
plot_df["urea_rate_timing"].value_counts(dropna=False)

urea_rate_timing
Urea 1kg/Are(V6 + V10)      401
Urea 1kg/Are(V6)            401
Urea 0.5Kg/Are(V6)          399
Urea1kg/Are(Planting+V6)    397
Name: count, dtype: int64

In [34]:
plot_df["urea_rate_timing"].nunique()

4

As expected, there are 4 treatment categories.

However, the next question is whether each field actually contains all four treatments

In [35]:
treatment_counts = (
    plot_df.groupby(["block_id", "season"])["urea_rate_timing"]
        .nunique()
)

treatment_counts.value_counts().sort_index()

urea_rate_timing
2      1
3      4
4    396
Name: count, dtype: int64

We can see then that:
- 396 fields have all four treatments,
- 4 have three treatments, and
- 1 has two treatments.

This is going to be essential for the experimental analysis.

### Treatment x plot structure check

We now go one step further and inspect how treatments are distributed within fields

What we want to establish is whether the experimental design described in the project dosumentation is actually reflected in the raw data.

In [36]:
plot_df.groupby(
    ["block_id", "season"]
)["urea_rate_timing"].value_counts()

block_id                              season  urea_rate_timing        
014d6b73d08a42c1be8990d884b17206      22A     Urea 0.5Kg/Are(V6)          1
                                              Urea1kg/Are(Planting+V6)    1
                                              Urea 1kg/Are(V6 + V10)      1
                                              Urea 1kg/Are(V6)            1
02a456dc-95f9-4e68-8187-f67031bcb563  21A     Urea 0.5Kg/Are(V6)          1
                                                                         ..
ff120793-f8d1-42dd-a0ba-d577cfd102e9  21A     Urea 1kg/Are(V6)            1
ff5a2b55-7060-4351-adf1-7de40fd9009c  21A     Urea 0.5Kg/Are(V6)          1
                                              Urea1kg/Are(Planting+V6)    1
                                              Urea 1kg/Are(V6 + V10)      1
                                              Urea 1kg/Are(V6)            1
Name: count, Length: 1598, dtype: int64

## Missingness Audit

While any research would be aiming to collect records for all variables, it is important to acknowledge that the process may still fail to collect some of the records.

In this section, our interest is to not only assess whether there are missing variables, but to also ascertain how much of the records are missing for each variable.

We'll be assessing the percentage of missing variables for each variable with missingness. 

A good rule of thumb is to not only flag out variables with missing records, but to also ascertain what percentage is missing. Possibly even visualize the missingness.

In [ ]:
for name,df in datasets.items():
    
    missing = (df.isnull().sum()/len(df))*100
    missing = missing.sort_values(ascending=False)
    
    print("\n")
    print(name)
    
    missing_var = missing[missing > 0]
    display(missing_var)
    if len(missing_var) > 0:
        subset_df = df[missing_var.index]
        fig, ax = plt.subplots(
            nrows = 1,
            ncols = 2,
            figsize = (16,6)
        )
        msno.matrix(subset_df,ax = ax[0])
        ax[0].set_title(f"{name}: Missing matrix")
        
        msno.heatmap(subset_df,ax = ax[1])
        ax[1].set_title(f"{name}: Missing Heatmap")
        plt.tight_layout()
        plt.show()    

## Determining Missingness Mechanism

Here, we are interested in identifying whether the missing data earlier identified are:

- Missing Completely At Randon (MCAR),
- Missing At Random (MAR), or
- Missing Not At Random (MNAR)

Since our primary interest is to investigate missingness by a universal variable like Mega Environment, for the field variables, we'll simply investigae why missingness occurs for the missing variables.

### Missingness in Field Data Frame

In [ ]:
field_missing_vars = ['trialfield_home_min','slope_pos','soil_depth','rockness',
                'cult_land','trial_follower_gender']

In [ ]:
for var in field_missing_vars:
    
    # Creating missing indicator
    field_df[f'{var}_missing'] = field_df[var].isnull()
    
    print(f"\n{'='*50}")
    print(f"missingness Pattern for: {var}")
    print(f"{'='*50}")
    
    output = (
        field_df.groupby(f'{var}_missing')['meg_env']
        .value_counts(dropna = False)
    )
    print(output)

The output above shows missingness by mega environment.

for instance, in terms of field position on hill:

-  72 records in highland mega environment are missing, and
 
-  37 records in transitional mega environments are missing

The above findings can further be represented in a more friendlier manner using plots as below.

In [ ]:
# Create a grid
fig, axes = plt.subplots(
    nrows = 3,
    ncols = 2,
    figsize = (16,14)
)
axes = axes.flatten()
for i, var in enumerate(field_missing_vars):
    # Missingness indicator
    field_df[f'{var}_missing'] = field_df[var].isnull()
    
    sns.countplot(
        data = field_df,
        x = 'meg_env',
        hue = f'{var}_missing',
        ax = axes[i]
)
axes[i].set_title(f'{var}')
axes[i].set_xlabel('Mega Environment')
axes[i].set_ylabel('Count')
axes[i].tick_params(axis = 'x', rotation = 45)

# Remove individual legends
axes[i].legend_.remove()

# Create one share legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    ['Present','Missing'],
    title = 'Status',
    loc = 'upper right'
)

plt.suptitle(
    'Missingness Pattern Accross Variables by Mega Environment',
    fontsize = 16
)

plt.tight_layout()
plt.subplots_adjust(top = 0.94)
plt.show()

The above column plots indicate a comparison of missing data by mega environment.

For instance, it illustrates that data on time from farmers' homes to the respective trial fields had the most data missing across all four mega environments. 
- On the same note, trial fields in highland mega environments emerge as the most likely to lack data on time from farmers' homes to trial fields.

The plots also illustrate that visual pebble content in trial fields was significantly missing across mega environments, especially in highland mega environments.

### Missingness in Plot Data Frame

For the Plot dataframe, since we don't have the Mega Environment as part of the dataframe, we'll need to create the smallest possible merge of the datasets, albeit temporarily, to investigate why missingness occurs.

In [ ]:

plot_audit = plot_df.merge(
    field_df[
        ['block_id','meg_env']
    ],
    on = 'block_id',
    how = 'left'
)

In [ ]:
plot_missing_vars = ['urea_V10_date','prior_season_crop','silking_date',
                     'compost_quality','R3_date']

In [ ]:
for var in plot_missing_vars:
    
    # Creating missing indicator
    plot_audit[f'{var}_missing'] = plot_audit[var].isnull()
    
    print(f"\n{'='*50}")
    print(f"missingness Pattern for: {var}")
    print(f"{'='*50}")
    
    output = (
        plot_audit.groupby(f'{var}_missing')['meg_env']
        .value_counts(dropna = False)
    )
    print(output)

In [ ]:
# Create a grid
fig, axes = plt.subplots(
    nrows = 3,
    ncols = 2,
    figsize = (16,14)
)
axes = axes.flatten()
for i, var in enumerate(plot_missing_vars):
    # Missingness indicator
    plot_audit[f'{var}_missing'] = plot_audit[var].isnull()
    
    sns.countplot(
        data = plot_audit,
        x = 'meg_env',
        hue = f'{var}_missing',
        ax = axes[i]
)
axes[i].set_title(f'{var}')
axes[i].set_xlabel('Mega Environment')
axes[i].set_ylabel('Count')
axes[i].tick_params(axis = 'x', rotation = 45)

# Remove individual legends
axes[i].legend_.remove()

# Create one share legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    ['Present','Missing'],
    title = 'Status',
    loc = 'upper right'
)

plt.suptitle(
    'Missingness Pattern Accross Variables by Mega Environment',
    fontsize = 16
)

plt.tight_layout()
plt.subplots_adjust(top = 0.94)
plt.show()

### Missingness in Soil Data Frame

Similarly, For the soil dataframe, since we only have the `plot_id` but no `block_id` or `meg_env` as variables, we'll have to perform a temporary merge so we can be able to investigate missingness by meg_env. 

For this, we'll need to merge the soil_df to the temporary plot_audit dataframe.

In [ ]:
soil_audit = soil_df.merge(
    plot_audit[
        ['plot_id','meg_env']
    ],
    on = 'plot_id',
    how = 'left'
)

In [ ]:
soil_missing_vars = ['tot_N','sand','org_C','pH']

In [ ]:
for var in soil_missing_vars:
    
    # Creating missing indicator
    soil_audit[f'{var}_missing'] = soil_audit[var].isnull()
    
    print(f"\n{'='*50}")
    print(f"missingness Pattern for: {var}")
    print(f"{'='*50}")
    
    output = (
        soil_audit.groupby(f'{var}_missing')['meg_env']
        .value_counts(dropna = False)
    )
    print(output)

In [ ]:
# Create a grid
fig, axes = plt.subplots(
    nrows = 2,
    ncols = 2,
    figsize = (16,10)
)
axes = axes.flatten()
for i, var in enumerate(soil_missing_vars):
    # Missingness indicator
    soil_audit[f'{var}_missing'] = soil_audit[var].isnull()
    
    sns.countplot(
        data = soil_audit,
        x = 'meg_env',
        hue = f'{var}_missing',
        ax = axes[i]
)
axes[i].set_title(f'{var}')
axes[i].set_xlabel('Mega Environment')
axes[i].set_ylabel('Count')
axes[i].tick_params(axis = 'x', rotation = 45)

# Remove individual legends
axes[i].legend_.remove()

# Create one share legend
handles, labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles,
    ['Present','Missing'],
    title = 'Status',
    loc = 'upper right'
)

plt.suptitle(
    'Missingness Pattern Accross Variables by Mega Environment',
    fontsize = 16
)

plt.tight_layout()
plt.subplots_adjust(top = 0.94)
plt.show()

## Duplicate Analysis

In this section, we are interested in flagging complete duplicate records. This should give insight into whether, for example, there were repeated measurements. 

In [ ]:
for name, df in datasets.items():
    
    print(
        f"{name}:",
        df.duplicated().sum()
    )

## Identifier integrity check

In [ ]:
field_df['block_id'].nunique()

In [ ]:
plot_df['plot_id'].nunique()

In [ ]:
plot_counts = plot_audit.groupby(
    'block_id'
).size()

plot_counts.value_counts()

In [ ]:
soil_df['plot_id'].nunique()

## Merge Simulation

After investigating the potential issues with our datasets, we can now merge the datasets.

However, before merging, we need to validate join keys

In [ ]:
field_plot_m = plot_df.merge(
    field_df,
    on = 'block_id',
    how = 'left',
    indicator = True
)

# Check to validate join
field_plot_m['_merge'].value_counts()

We further check for row inflation before we can ascertain that the merged dataset would retain it's original integrity.

If the master dataframe retains its integrity, we move forward to merge with the soil_df

In [ ]:
print(
    "Before:",
    len(plot_df)
)

print(
    "After:",
    len(field_plot_m)
)

In [ ]:
field_plot_m.head()

## Outlier Analysis